# XGBoost for pMI prediction

Fit PCA once, then reuse its features while changing XGBoost hyperparameters.
Run with the **Python (MCPP)** kernel. The project dependencies include XGBoost.

`motion_only=True` means **heart XYZ coordinates from all frames, without demographics or anatomical labels**. It does not calculate displacement. By default, demographics are appended **after** PCA.

The data and PCA preparation cells reuse their existing results when rerun. Change the PCA settings or set `REFIT_PCA=True` to rebuild the features; changing only XGBoost settings never refits PCA. Restarting the kernel clears this in-memory cache.


In [ ]:
from contextlib import redirect_stdout
from importlib import reload
from pathlib import Path
import io

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

from mcpp.data_loader import DataSet
import mcpp.xgboost as xgboost_module
reload(xgboost_module)
from mcpp.xgboost import (
    prepare_xgboost_data,
    train_xgboost,
    evaluate_xgboost,
    calibrate_xgboost_threshold,
    load_xgboost_feature_cache,
    save_xgboost_feature_cache,
)


## Load patients once

The existing loader makes stratified patient-level train/validation/test splits. Every frame from a patient stays in the same split. The validation set supports model selection; the test set is reserved for the final evaluation.


In [10]:
if "dataset" not in globals():
    # Suppress the loader's per-patient printout.
    with redirect_stdout(io.StringIO()):
        dataset = DataSet("training_data")

print({name: len(getattr(dataset, name).patients) for name in ("train", "val", "test")})


{'train': 540, 'val': 180, 'test': 180}


## Prepare PCA features once

PCA is fitted on training patients only. Validation and test patients use that same fitted model. The first run processes all points and may take some time.

With `motion_only=True`, use `include_demographics=False` for a heart-data-only classifier. Setting `motion_only=False` restores the original combined PCA input; demographics then already enter PCA and are not appended again.


In [ ]:
PCA_CONFIG = {
    "n_components": 100,
    "motion_only": True,
    "include_demographics": True,
}
REFIT_PCA = False
PROJECT_DIR = dataset.directory.parent.parent
PCA_CACHE_DIR = PROJECT_DIR / "results" / "xgboost_tuning"
preparation_key = (id(dataset), tuple(sorted(PCA_CONFIG.items())))
same_in_memory_features = (
    globals().get("_preparation_key") == preparation_key
    and "prepared" in globals()
)

if not REFIT_PCA:
    cached_prepared, cache_manifest = load_xgboost_feature_cache(
        PCA_CACHE_DIR,
        dataset,
        n_components=PCA_CONFIG["n_components"],
        motion_only=PCA_CONFIG["motion_only"],
        include_demographics=PCA_CONFIG["include_demographics"],
    )
else:
    cached_prepared, cache_manifest = None, None

if cached_prepared is not None:
    prepared = cached_prepared
    print("Loaded matching 100-component PCA features from disk. PCA was not refitted.")
elif not REFIT_PCA and same_in_memory_features:
    prepared = globals()["prepared"]
    cache_manifest = save_xgboost_feature_cache(
        PCA_CACHE_DIR, dataset, prepared, **PCA_CONFIG,
    )
    print("Reused the existing fitted PCA and saved its reusable feature cache.")
else:
    prepared = prepare_xgboost_data(dataset, **PCA_CONFIG)
    cache_manifest = save_xgboost_feature_cache(
        PCA_CACHE_DIR, dataset, prepared, **PCA_CONFIG,
    )
    print("Fitted PCA once and saved a reusable feature cache.")

_preparation_key = preparation_key
experiment_history = globals().get("experiment_history", []) if same_in_memory_features else []
REFIT_PCA = False

print("Train / validation / test:", prepared.X_train.shape, prepared.X_val.shape, prepared.X_test.shape)
print("Training variance retained:",
      f"{cache_manifest['explained_variance_ratio_sum']:.2%}")


## Change XGBoost hyperparameters here

These settings are the best result from an expanded randomized search of **300 configurations per PCA size** (10, 25, 50, 100, 200, 300, and 540 PCs), using five-fold patient-level cross-validation and macro-average F1 as the objective. The selected 100-PC model has validation macro-F1 **0.849** at its training-fold-calibrated cutoff.

The threshold is recalibrated using training patients only whenever you rerun the model cell. Changing XGBoost parameters does not refit PCA. The validation set helped select PCA size, so its score is a model-selection result; the test split remains untouched.


In [ ]:
XGB_PARAMS = {
    "colsample_bytree": 0.24489483601472165,
    "gamma": 0.02955209235202888,
    "learning_rate": 0.04509821930882766,
    "max_bin": 128,
    "max_depth": 5,
    "min_child_weight": 0.07863688145546184,
    "reg_alpha": 0.00011674193588234575,
    "reg_lambda": 2.0184436705608353,
    "subsample": 0.9019811459609137,
    "grow_policy": "lossguide",
    "max_leaves": 115,
    "max_delta_step": 5,
    "n_estimators": 180,
    "random_state": 20261015,
    "early_stopping_rounds": null,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "n_jobs": 4
}


In [ ]:
# Recalibrate after every parameter change using training patients only.
CLASSIFICATION_THRESHOLD = calibrate_xgboost_threshold(
    prepared.X_train, prepared.y_train, objective="macro_f1", **XGB_PARAMS,
)
classifier = train_xgboost(
    prepared.X_train, prepared.y_train,
    prepared.X_val, prepared.y_val,
    **XGB_PARAMS,
)

validation_metrics = evaluate_xgboost(
    classifier, prepared.X_val, prepared.y_val,
    threshold=CLASSIFICATION_THRESHOLD,
)
experiment_history.append({
    "run": len(experiment_history) + 1,
    "classification_threshold": CLASSIFICATION_THRESHOLD,
    **XGB_PARAMS,
    **validation_metrics,
})
print(f"Training-calibrated cutoff (max macro-F1): {CLASSIFICATION_THRESHOLD:.3f}")
display(pd.Series(validation_metrics, name="validation"))
display(pd.DataFrame(experiment_history).sort_values("roc_auc", ascending=False))


## Expanded PCA dimension and hyperparameter search

For each PCA size, I evaluated **300 randomized XGBoost configurations** with five-fold patient-level cross-validation. Search ranges covered learning rates 0.001–0.5, depths 1–16, child weights 0.01–300, subsampling 0.3–1.0, feature subsampling 0.1–1.0, L1 regularization 0–1,000, L2 1e-6–1e4, gamma 0–100, both growth policies, leaf counts 8–256, max-bin 32–1,024, and max-delta-step 0/1/5/10. Each fold calibrated its classification cutoff on a separate training-only subset to maximize macro-F1.

| PCs | CV macro-F1 | Validation macro-F1 | Validation AUC | Validation balanced accuracy | pMI detected | Cutoff |
| ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 10 | 0.800 | 0.794 | 0.854 | 0.795 | 77 / 89 | 0.412 |
| 25 | 0.827 | 0.800 | 0.889 | 0.800 | 75 / 89 | 0.431 |
| 50 | 0.842 | 0.822 | 0.899 | 0.822 | 71 / 89 | 0.467 |
| 100 | 0.861 | 0.849 | 0.930 | 0.849 | 71 / 89 | 0.498 |
| 200 | 0.844 | 0.783 | 0.871 | 0.783 | 67 / 89 | 0.422 |
| 300 | 0.842 | 0.833 | 0.926 | 0.833 | 75 / 89 | 0.480 |
| 540 | 0.831 | 0.429 | 0.858 | 0.534 | 9 / 89 | 0.486 |

The notebook is set to **100 PCs**, the highest validation macro-F1 among these candidates. The validation set was used to select the dimension, so these metrics are selection results rather than an unbiased final estimate. The test set was not scored. Search details are in `results/xgboost_tuning/pca_dimension_comparison_macro_f1.json` and the corresponding `macro_f1_broad_v2_trials_*pc.csv` files.


## Inspect the latest model

The table above records earlier runs; `classifier` always refers to the latest run. To select an earlier configuration, copy its settings into `XGB_PARAMS` and retrain. This still reuses the same PCA.


In [22]:
validation_probabilities = classifier.predict_proba(prepared.X_val)[:, 1]
validation_predictions = (validation_probabilities >= 0.5).astype(int)
print(classification_report(
    prepared.y_val, validation_predictions,
    labels=[0, 1], target_names=["healthy", "pMI"], zero_division=0,
))
ConfusionMatrixDisplay.from_predictions(
    prepared.y_val, validation_predictions,
    labels=[0, 1], display_labels=["healthy", "pMI"],
)
plt.show()


              precision    recall  f1-score   support

     healthy       0.66      1.00      0.80        91
         pMI       1.00      0.48      0.65        89

    accuracy                           0.74       180
   macro avg       0.83      0.74      0.72       180
weighted avg       0.83      0.74      0.73       180

Confusion matrix (healthy, pMI): [[91, 0], [46, 43]]


In [ ]:
feature_importance = pd.Series(
    classifier.feature_importances_, index=prepared.feature_names,
    name="XGBoost importance",
).sort_values(ascending=False)
display(feature_importance.head(20))


## Final test evaluation — after choosing settings

Leave this disabled while experimenting. Once your configuration is fixed and the selected model has been trained above, enable it and evaluate once. Do not use test scores to choose hyperparameters.


In [16]:
RUN_FINAL_TEST = False

if RUN_FINAL_TEST:
    display(pd.Series(
        evaluate_xgboost(classifier, prepared.X_test, prepared.y_test),
        name="held-out test",
    ))
else:
    print("Test evaluation is disabled while tuning.")


Test evaluation is disabled while tuning.
